# Vega — successive query revisions reuse what they share

Exploratory analysis is a sequence of near-identical queries, each normally starting
from nothing. This materialises the reusable parts as a query runs, so the next
revision starts from the deepest point the two still share.

In [ ]:
import os, sys, glob

ROOT = os.environ.get("BIGASTERISK_HOME") or os.path.abspath("..")

# Jars: a source checkout has them under modules/*/target, the Docker image under jars/.
JARS = sorted(glob.glob(f"{ROOT}/modules/*/target/scala-2.13/bigasterisk-*.jar")) \
    or sorted(glob.glob(f"{ROOT}/jars/bigasterisk-*.jar"))
if not JARS:
    raise SystemExit("No BigAsterisk jars found. Run: bin/sbt package")

FASTUTIL_JAR = os.environ.get("FASTUTIL_JAR") or next(iter(sorted(
    glob.glob(f"{ROOT}/jars/fastutil*.jar")
    + glob.glob(os.path.expanduser("~/Library/Caches/Coursier/**/fastutil-8.5.15.jar"), recursive=True)
    + glob.glob(os.path.expanduser("~/.cache/coursier/**/fastutil-8.5.15.jar"), recursive=True)
)), None)
if not FASTUTIL_JAR:
    raise SystemExit("fastutil jar not found. Run: bin/sbt package")

SPARK_JARS = ",".join(JARS + [FASTUTIL_JAR])
DATA = f"{ROOT}/examples/data"
sys.path.insert(0, f"{ROOT}/python")

## The data

Twelve orders across three customers. One of them, `o8`, is an outlier at
`99999` — every notebook here uses it as the thing to find.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import bigasterisk

spark = (bigasterisk.configure(SparkSession.builder)
    .master("local[2]")
    .appName("vega-notebook")
    .config("spark.jars", SPARK_JARS)
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

orders = spark.read.schema("oid STRING, cid STRING, amount INT").csv(f"{DATA}/orders.txt")
customers = spark.read.schema("cid STRING, name STRING").csv(f"{DATA}/customers.txt")
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

orders.show()

## Run one query, then a revision of it

In [ ]:
vega = bigasterisk.vega(spark)
vega.clear()

first = vega.run("SELECT cid, amount FROM orders WHERE amount > 100")
first.df.collect()
print("first run reused:", first.reused)
print("first run materialised:", first.materialized)

In [ ]:
second = vega.run("SELECT cid, SUM(amount) AS total FROM orders "
                  "WHERE amount > 100 GROUP BY cid")
print("revision reused:", second.reused)
print("reuse ratio: %.0f%%" % (second.reuse_ratio * 100))
second.df.show()

## Check

Reuse must never change the answer.

In [ ]:
expected = {(r["cid"], r["total"]) for r in spark.sql(
    "SELECT cid, SUM(amount) AS total FROM orders WHERE amount > 100 GROUP BY cid"
).collect()}
actual = {(r["cid"], r["total"]) for r in second.df.collect()}
assert actual == expected, (actual, expected)
assert len(second.reused) > 0
vega.clear()
print("OK")